In [ ]:
!pip uninstall -y triton bitsandbytes
!pip install -q bitsandbytes --prefer-binary
!pip install -q \
    transformers==4.44.0 \
    peft==0.12.0 \
    trl==0.10.1 \
    accelerate==0.33.0 \
    wandb \
    sentencepiece \
    datasets

Found existing installation: triton 3.6.0
Uninstalling triton-3.6.0:
  Successfully uninstalled triton-3.6.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.4/296.4 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.1/280.1 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━

In [ ]:
import os
from google.colab import userdata
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

In [ ]:
import os
import json
import torch
import wandb
from pathlib import Path
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel,
)
from trl import SFTTrainer, SFTConfig

print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")
print(f"GPU              : {torch.cuda.get_device_name(0)}")
print(f"GPU Memory       : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version  : 2.10.0+cu128
CUDA available   : True
GPU              : NVIDIA A100-SXM4-80GB
GPU Memory       : 85.1 GB


In [ ]:
# ── Paths ───────────────────────────────────────────────────
DATA_DIR    = Path("/content/drive/MyDrive/Legal_Analyser_Data/processed")
OUTPUT_DIR  = Path("/content/phi3-legal")
ADAPTER_DIR = Path("/content/phi3-legal-adapter")
MERGED_DIR  = Path("/content/phi3-legal-merged")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
MERGED_DIR.mkdir(parents=True, exist_ok=True)

# ── Model ───────────────────────────────────────────────────
MODEL_ID       = "microsoft/Phi-3-mini-4k-instruct"
MAX_SEQ_LENGTH = 2048

# ── LoRA ────────────────────────────────────────────────────
LORA_R         = 8
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.15      # increased from 0.05
TARGET_MODULES = [
    "qkv_proj",
    "o_proj",
    "gate_up_proj",
    "down_proj",
]

# ── Training ────────────────────────────────────────────────
EPOCHS           = 3
BATCH_SIZE       = 4       # increased for A100
GRAD_ACCUM_STEPS = 4       # effective batch = 16
LEARNING_RATE    = 5e-5    # reduced from 2e-4
WARMUP_RATIO     = 0.1
LR_SCHEDULER     = "cosine"
MAX_GRAD_NORM    = 0.2     # tighter clipping

# ── W&B ─────────────────────────────────────────────────────
WANDB_PROJECT = "legal-contract-analyzer"
WANDB_RUN     = "phi3-mini-qlora-v2"

print("Configuration loaded ✓")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")

Configuration loaded ✓
Effective batch size: 16


In [ ]:
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN,
    config={
        "model_id"      : MODEL_ID,
        "lora_r"        : LORA_R,
        "lora_alpha"    : LORA_ALPHA,
        "lora_dropout"  : LORA_DROPOUT,
        "epochs"        : EPOCHS,
        "batch_size"    : BATCH_SIZE,
        "grad_accum"    : GRAD_ACCUM_STEPS,
        "learning_rate" : LEARNING_RATE,
        "max_seq_length": MAX_SEQ_LENGTH,
        "max_grad_norm" : MAX_GRAD_NORM,
    }
)
print(f"W&B initialized ✓ — Run: {WANDB_RUN}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: khushpatel9235 (khushpatel9235-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


W&B initialized ✓ — Run: phi3-mini-qlora-v2


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
print("Loading datasets...")

dataset = load_dataset(
    "json",
    data_files={
        "train" : str(DATA_DIR / "train.jsonl"),
        "val"   : str(DATA_DIR / "val.jsonl"),
    }
)

print(f"Train examples : {len(dataset['train'])}")
print(f"Val examples   : {len(dataset['val'])}")
print(f"\nSample (first 300 chars):")
print(dataset["train"][0]["text"][:300])

Loading datasets...


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Train examples : 5109
Val examples   : 510

Sample (first 300 chars):
<|system|>
You are a legal contract analyst. Analyze the contract text and detect if the specified clause is present. If present, extract the relevant text and explain it in plain English. If not present, state 'NOT PRESENT'.
<|end|>
<|user|>
Contract text:
"T I P A S G S G T I P A x x D x I P A x I


In [ ]:
print(f"Loading {MODEL_ID} in 4-bit...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
    torch_dtype=torch.bfloat16,
    attn_implementation="eager",
)
model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=False,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Base model loaded ✓")
print(f"Model dtype: {model.dtype}")

Loading microsoft/Phi-3-mini-4k-instruct in 4-bit...


config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/599 [00:00<?, ?B/s]

Base model loaded ✓
Model dtype: torch.bfloat16


In [ ]:
print("Applying QLoRA adapters...")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("QLoRA adapters applied ✓")

Applying QLoRA adapters...
trainable params: 12,582,912 || all params: 3,833,662,464 || trainable%: 0.3282
QLoRA adapters applied ✓


In [ ]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER,
    max_grad_norm=MAX_GRAD_NORM,
    fp16=False,
    bf16=True,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    logging_steps=25,
    report_to="wandb",
    run_name=WANDB_RUN,
    dataloader_pin_memory=False,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["val"],
    args=sft_config,
)

print("SFTTrainer built ✓")
print(f"Training on    : {len(dataset['train'])} examples")
print(f"Validating on  : {len(dataset['val'])} examples")
print(f"Effective batch: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Steps per epoch: {len(dataset['train']) // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
print("\nStarting training...")

trainer.train()

print("\nTraining complete ✓")

Map:   0%|          | 0/5109 [00:00<?, ? examples/s]

Map:   0%|          | 0/510 [00:00<?, ? examples/s]

SFTTrainer built ✓
Training on    : 5109 examples
Validating on  : 510 examples
Effective batch: 16
Steps per epoch: 319

Starting training...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
You are not running the flash-attention implementation, expect numerical differences.


Step,Training Loss,Validation Loss
100,1.133500,1.093845
200,0.917400,0.953205
300,0.872400,0.932276
400,0.811500,0.935015
500,0.777700,0.937544
600,0.751500,0.961099
700,0.729100,0.981175
800,0.717500,0.980903
900,0.703800,0.983244


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


Training complete ✓


In [ ]:
print("Saving LoRA adapter...")

trainer.model.save_pretrained(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))

print(f"Adapter saved ✓")
for f in ADAPTER_DIR.iterdir():
    size = f.stat().st_size / 1024
    print(f"  {f.name} — {size:.1f} KB")

Saving LoRA adapter...


NameError: name 'trainer' is not defined

In [ ]:
print("Merging adapter into base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager",
)

merged_model = PeftModel.from_pretrained(base_model, str(ADAPTER_DIR))
merged_model = merged_model.merge_and_unload()

merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))

print(f"Merged model saved ✓")

wandb.finish()
print("W&B run finished ✓")

Merging adapter into base model...


NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
# Save merged model to Drive so it persists after session ends
import shutil

DRIVE_SAVE = "/content/drive/MyDrive/legal-contract-training/phi3-legal-merged"

print(f"Copying merged model to Google Drive...")
shutil.copytree(str(MERGED_DIR), DRIVE_SAVE, dirs_exist_ok=True)
print(f"Saved to Drive ✓ — {DRIVE_SAVE}")

Copying merged model to Google Drive...


NameError: name 'MERGED_DIR' is not defined

In [ ]:
import os

MERGED_DIR = "/content/drive/MyDrive/legal-contract-training/phi3-legal-merged"

print("Merged model files:")
total_size = 0
for f in sorted(os.listdir(MERGED_DIR)):
    size = os.path.getsize(f"{MERGED_DIR}/{f}") / 1024 / 1024
    total_size += size
    print(f"  {f} — {size:.1f} MB")

print(f"\nTotal size: {total_size:.1f} MB")

Merged model files:
  added_tokens.json — 0.0 MB
  config.json — 0.0 MB
  generation_config.json — 0.0 MB
  model-00001-of-00002.safetensors — 4742.1 MB
  model-00002-of-00002.safetensors — 2546.0 MB
  model.safetensors.index.json — 0.0 MB
  special_tokens_map.json — 0.0 MB
  tokenizer.json — 1.8 MB
  tokenizer.model — 0.5 MB
  tokenizer_config.json — 0.0 MB

Total size: 7290.4 MB
